In [1]:
import mlflow

# mlflow.set_tracking_uri("sqlite:///mlflow.db")
import mlflow.sklearn

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

# from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

df = pd.read_csv("../data/raw/creditcard.csv")

X = df.drop("Class", axis=1)
y = df["Class"]

X_train,X_test,y_train,y_test= train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(random_state = 42)
# rf.fit(X_train,y_train)

# Hyperparameter Grid

param_grid ={
    "n_estimators":[100, 200],
    "max_depth":[10, None],
    "min_samples_split":[2],
    "min_samples_leaf": [1]
}

grid = GridSearchCV(
    estimator = rf,
    param_grid = param_grid,
    cv = 5,
    scoring = "f1",
    n_jobs = -1,
    verbose = 2
)

grid.fit(X_train, y_train)

# print(grid.best_params_)

best_params = grid.best_params_
print(best_params)

best_rf = grid.best_estimator_

best_pred =best_rf.predict(X_test)
accuracy = accuracy_score(y_test, best_pred)
precision = precision_score(y_test, best_pred)
recall = recall_score(y_test, best_pred)
f1 = f1_score(y_test, best_pred)

# Log Parameters

# mlflow.log_param(
#     "n_estimators",
#     200
# )

# mlflow.log_param(
#     "max_depth",
#     10
# )
best_params = grid.best_params_

for key, value in best_params.items():
    mlflow.log_param(key, value)

mlflow.end_run()

mlflow.start_run()

# Log Metrics

mlflow.log_metric(
    "accuracy",
    accuracy
)

mlflow.log_metric(
    "precision",
    precision
)

mlflow.log_metric(
    "recall",
    recall
)

mlflow.log_metric(
    "f1",
    f1
)

# Log Model

mlflow.sklearn.log_model(
    best_rf,
    "random_forest_model"
)

mlflow.end_run()
print("MLflow run completed successfully!")

Fitting 5 folds for each of 4 candidates, totalling 20 fits
{'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}


2026/07/09 21:26:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/09 21:26:40 INFO mlflow.store.db.utils: Updating database tables
2026/07/09 21:26:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MLflow run completed successfully!
